# Task 2: Technical Analysis
This notebook implements technical indicators (SMA, EMA, RSI, MACD) for stock price data, using both manual calculations and PyNance.

In [ ]:
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import os
import pynance as pn

def calculate_rsi(data, window=14):
    delta = data.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

def calculate_macd(data, slow=26, fast=12, signal=9):
    exp1 = data.ewm(span=fast, adjust=False).mean()
    exp2 = data.ewm(span=slow, adjust=False).mean()
    macd = exp1 - exp2
    signal_line = macd.ewm(span=signal, adjust=False).mean()
    return macd, signal_line

ticker = 'AAPL'
stock_data = yf.download(ticker, start='2023-01-01', end='2023-05-01')
stock_data['SMA_20'] = stock_data['Close'].rolling(window=20).mean()
stock_data['EMA_20'] = stock_data['Close'].ewm(span=20, adjust=False).mean()
stock_data['RSI_14'] = calculate_rsi(stock_data['Close'])
stock_data['MACD'], stock_data['MACD_Signal'] = calculate_macd(stock_data['Close'])
stock_data.tail()

## 1. Using PyNance for Additional Metrics
PyNance provides high-level financial functions.

In [ ]:
# Example: Getting daily returns using pynance (if supported by the version)
try:
    returns = pn.data.get_returns(stock_data['Close'])
    print("Daily Returns (first 5):")
    print(returns.head())
except AttributeError:
    print("pn.data.get_returns not available in this version of pynance.")

## 2. Visualizing Indicators

In [ ]:
plt.figure(figsize=(12, 10))
plt.subplot(3, 1, 1)
plt.plot(stock_data['Close'], label='Close')
plt.plot(stock_data['SMA_20'], label='SMA 20')
plt.legend()

plt.subplot(3, 1, 2)
plt.plot(stock_data['RSI_14'], label='RSI 14', color='purple')
plt.axhline(70, color='red', linestyle='--')
plt.axhline(30, color='green', linestyle='--')
plt.legend()

plt.subplot(3, 1, 3)
plt.plot(stock_data['MACD'], label='MACD')
plt.plot(stock_data['MACD_Signal'], label='Signal')
plt.legend()
plt.show()